# Stage 0 — TF-IDF Topic Scoring
**[STUDENT VERSION — fill in the blanks]**

Stage này biến mô tả văn bản của mỗi điểm đến thành **điểm liên quan theo từ vựng** đối với 8 chủ đề: `beach`, `history`, `food`, `nature`, `adventure`, `culture`, `relax`, `photo`.

## Sau stage này, bạn có thể

- giải thích TF, IDF và lý do từ hiếm có khả năng phân biệt tốt hơn;
- biểu diễn mô tả và câu mẫu trong cùng một không gian vector;
- dùng cosine similarity để đo mức độ giống nhau về hướng;
- chuẩn hóa từng cột điểm về `[0, 1]` và nêu được giới hạn của phép chuẩn hóa;
- dùng sanity check để phát hiện kết quả bất thường trước khi chuyển sang stage tiếp theo.

## Bức tranh toàn bộ pipeline

```text
descriptions + anchors
        ↓ fit chung một TF-IDF vectorizer
các vector trong cùng vocabulary space
        ↓ cosine similarity
ma trận raw: destination × topic (38 × 8)
        ↓ min-max riêng từng cột
ma trận score [0, 1] → CSV → sanity check top 5
```

`anchors` là các câu mẫu do con người viết để đại diện cho từng chủ đề. Có thể xem chúng như **prototype/weak label** giúp gán điểm ban đầu, không phải nhãn đúng tuyệt đối và cũng không phải semantic embedding đã hiểu sâu ý nghĩa câu. Vì TF-IDF chủ yếu dựa vào token và cụm token trùng nhau, hai câu đồng nghĩa nhưng dùng từ rất khác vẫn có thể nhận cosine thấp.

- Input: `stage0_descriptions.csv` — 38 Vietnamese destination descriptions.
- Input: `stage0_anchors.csv` — 8 semantic feature anchors.
- Output: normalized semantic score matrix (38 × 8).

> 💡 Các cell có `# TODO` là phần học viên cần hoàn thành. Hãy đọc hình dạng dữ liệu mong đợi trước khi viết code.

In [1]:
import csv
import re
from pathlib import Path

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
BASE_DIR = Path('.')
DESCRIPTIONS_CSV = BASE_DIR / 'stage0_descriptions.csv'
ANCHORS_CSV = BASE_DIR / 'stage0_anchors.csv'

def read_csv_rows(path):
    with path.open('r', encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

description_rows = read_csv_rows(DESCRIPTIONS_CSV)
anchor_rows      = read_csv_rows(ANCHORS_CSV)

places       = [row['place']       for row in description_rows]
provinces    = [row['province']    for row in description_rows]
descriptions = [row['description'] for row in description_rows]
features     = [row['feature']     for row in anchor_rows]
anchors      = [row['anchor_text'] for row in anchor_rows]


FileNotFoundError: [Errno 2] No such file or directory: 'stage0_descriptions.csv'

In [ ]:
WORD_RE = re.compile(r"\b[\wÀ-ỹà-ỹĐđ]+\b", re.UNICODE)
word_counts = [len(WORD_RE.findall(text)) for text in descriptions]
bad_counts  = [(p, c) for p, c in zip(places, word_counts) if c < 150 or c > 250]
if bad_counts:
    raise ValueError(f'Descriptions outside 150-250 word range: {bad_counts}')

print(f'Destination count: {len(descriptions)}')
print(f'Word count range: {min(word_counts)}-{max(word_counts)}')


Destination count: 38
Word count range: 192-220


## 🔧 TODO 1 — TF-IDF Vectorizer

Đây là bước biến văn bản thành vector số để máy có thể so sánh.

### 1. TF-IDF đo điều gì?

- **TF (term frequency):** token xuất hiện nhiều trong một văn bản thì có nhiều bằng chứng hơn cho nội dung của văn bản đó.
- **IDF (inverse document frequency):** token xuất hiện trong quá nhiều văn bản thì ít có khả năng phân biệt.

Với `sublinear_tf=True`, scikit-learn thay tần suất thô bằng:

```text
tf_sublinear(t, d) = 1 + log(tf(t, d))    nếu tf(t, d) > 0
```

Đây là `1 + log(tf)`, **không phải** `log(1 + tf)`. Phép log làm giảm ảnh hưởng của việc một token bị lặp lại quá nhiều. Với thiết lập làm trơn IDF mặc định, trực giác của công thức là:

```text
idf(t) = log((1 + N) / (1 + df(t))) + 1
tfidf(t, d) = tf_sublinear(t, d) × idf(t)
```

Trong đó `N` là số văn bản trong corpus và `df(t)` là số văn bản có token `t`.

### 2. Vì sao phải fit chung descriptions và anchors?

Mỗi vị trí trong vector phải đại diện cho cùng một token. Khi fit chung `descriptions + anchors`, cả hai phía dùng chung vocabulary và cùng cách tính IDF. Nếu fit hai vectorizer riêng, cột số 10 ở hai ma trận có thể đại diện cho hai token khác nhau; khi đó dot product và cosine không còn ý nghĩa.

Anchor chỉ đóng vai trò câu đại diện. Điểm cao có nghĩa là mô tả và anchor có nhiều token/cụm token quan trọng giống nhau; nó không chứng minh điểm đến “thật sự thuộc” chủ đề theo một nhãn khách quan.

### 3. Cosine similarity — nhìn vào hướng, không chỉ độ dài

```text
cosine(a, b) = (a · b) / (||a|| × ||b||)
```

Hãy tưởng tượng mỗi vector là một mũi tên. Hai mũi tên cùng nhấn mạnh các token giống nhau sẽ tạo góc nhỏ và cosine gần `1`; ít token quan trọng chung thì cosine gần `0`. Vì vector TF-IDF ở đây không âm, raw cosine thường nằm trong `[0, 1]`.

Ví dụ trực giác: mô tả có các cụm “bãi biển”, “cát trắng”, “nghỉ dưỡng” sẽ gần anchor `beach` hơn một mô tả chỉ nói về “di tích” và “bảo tàng”. Đây vẫn là so khớp từ vựng, không phải hiểu nghĩa sâu như mô hình embedding.

Tham số cần dùng:
- `lowercase=True`
- `sublinear_tf=True` — dùng `1 + log(tf)` thay vì TF thô
- `ngram_range=(1, 2)` — unigram + bigram
- `token_pattern=r"(?u)\b\w+\b"`

Unigram bắt token đơn như “biển”, “núi”; bigram giữ thêm một phần ngữ cảnh trong cụm như “cát trắng”, “di sản”. Với tiếng Việt, cách tách theo khoảng trắng vẫn gần với âm tiết hơn là một bộ tách từ chuyên dụng, nên đây là một baseline dễ hiểu chứ chưa phải cách biểu diễn hoàn hảo.

**Tự kiểm tra trước khi sang TODO 2:** corpus phải chứa cả descriptions lẫn anchors; hai nhóm vector phải có cùng số cột; ma trận raw phải có một hàng cho mỗi điểm đến và một cột cho mỗi feature.

In [ ]:
# TODO 1: Tạo vectorizer, fit_transform corpus, tính raw cosine similarity

# Bước 1: Tạo corpus = descriptions + anchors
corpus = None  # ← thay None bằng code của bạn

# Bước 2: Khởi tạo TfidfVectorizer với đúng tham số
vectorizer = None  # ← thay None bằng TfidfVectorizer(...)

# Bước 3: fit_transform corpus → ma trận X
X = None  # ← thay None bằng vectorizer.fit_transform(corpus)

# Bước 4: Tính raw cosine similarity
# X[:len(descriptions)] = description vectors
# X[len(descriptions):] = anchor vectors
raw = None  # ← thay None bằng cosine_similarity(...)

print(f'Vocabulary size: {len(vectorizer.get_feature_names_out())}')
print(f'Raw matrix shape: {raw.shape}')  # phải là (38, 8)


## 🔧 TODO 2 — Min-Max Normalization per Feature

Các cột raw cosine có thể có khoảng giá trị khác nhau vì nội dung và độ đặc trưng của từng anchor khác nhau. Ta min-max **riêng từng cột** để đưa mỗi cột về cùng thang số `[0, 1]` trong tập dữ liệu hiện tại.

Công thức:
```
normalized[i, j] = (raw[i, j] - min_j) / (max_j - min_j)
```

Ví dụ, raw score của một feature là `[0.02, 0.07, 0.12]`. Khi đó `min=0.02`, `max=0.12`, nên kết quả là `[0.0, 0.5, 1.0]`. Giá trị `1.0` chỉ nói rằng đây là điểm cao nhất **trong cột và tập dữ liệu đang xét**.

### Những điều cần nhớ

- Score sau min-max **không phải xác suất**: `0.8` không có nghĩa là xác suất thuộc chủ đề bằng 80%.
- Cùng thang `[0, 1]` không tự động làm các feature “công bằng”, quan trọng như nhau hay hết thiên lệch; chất lượng vẫn phụ thuộc mô tả, anchor và dữ liệu.
- Min-max nhạy với điểm ngoại lệ. Khi thêm điểm đến mới hoặc tính lại trên corpus khác, `min`/`max` có thể đổi và score cũ không còn so sánh trực tiếp được.
- Nếu `max_j == min_j`, mẫu số bằng 0; cần xử lý riêng để tránh `NaN` hoặc lỗi chia cho 0.

**Tự kiểm tra sau TODO 2:** shape của `scores` phải giữ nguyên như `raw`; mọi giá trị phải hữu hạn và nằm trong `[0, 1]`; không được xuất hiện `NaN` hay `inf`.

In [ ]:
# TODO 2: Per-feature min-max normalization

mins   = None  # ← min của mỗi cột (axis=0)
maxs   = None  # ← max của mỗi cột (axis=0)
scores = np.zeros_like(raw)

for j in range(raw.shape[1]):
    denom = None  # ← maxs[j] - mins[j]
    # TODO: điền công thức normalize vào đây
    scores[:, j] = None  # ← (raw[:, j] - mins[j]) / denom if denom > 0 else 0.0

print(f'Score matrix shape: {scores.shape}')
print(f'Score range: {scores.min():.3f} – {scores.max():.3f}')  # phải là 0.0 – 1.0


In [ ]:
def write_csv(path, rows, headers):
    with path.open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        writer.writerows(rows)

score_rows = []
raw_rows   = []
for i, place in enumerate(places):
    base = {'place': place, 'province': provinces[i]}
    score_rows.append({**base, **{feature: f'{scores[i, j]:.3f}'  for j, feature in enumerate(features)}})
    raw_rows.append(  {**base, **{feature: f'{raw[i, j]:.6f}'    for j, feature in enumerate(features)}})

headers = ['place', 'province'] + features
write_csv(BASE_DIR / 'stage0_tfidf_scores_rerun.csv', score_rows, headers)
write_csv(BASE_DIR / 'stage0_raw_cosine_rerun.csv',   raw_rows,   headers)
print('Saved.')


## 🔧 TODO 3 — Sanity Check

In top 5 điểm đến của mỗi feature là một **sanity check**: phép kiểm tra nhanh xem pipeline có tạo ra kết quả hợp lý ở mức trực giác hay không. Ví dụ, ta kỳ vọng danh sách `beach` có những địa điểm nổi tiếng về biển như Biển Nhật Lệ hoặc Lăng Cô. Đây là kỳ vọng để điều tra, không phải luật bắt buộc hay nhãn đúng tuyệt đối.

Sanity check có thể phát hiện lỗi rõ ràng như đảo chiều sort, chọn nhầm cột, toàn bộ score bằng nhau hoặc anchor không khớp từ vựng. Tuy nhiên, nó **không thay thế đánh giá mô hình**. Muốn kết luận chất lượng, cần một tập đánh giá độc lập có nhận định của con người và metric phù hợp như Precision@K hoặc nDCG@K.

### Checklist tự đánh giá

- Mỗi feature in đúng 5 địa điểm hợp lệ và không lặp.
- Thứ tự đi từ score cao xuống thấp.
- Có thể giải thích ít nhất một kết quả cao bằng token/bigram chung giữa mô tả và anchor.
- Nếu kết quả bất ngờ, kiểm tra lần lượt dữ liệu đầu vào, vocabulary, lát cắt hai nhóm vector, chiều của ma trận và bước sort.

### Câu hỏi suy ngẫm

1. Vì sao không thể fit descriptions và anchors bằng hai vectorizer độc lập?
2. Vì sao score `1.0` sau min-max không phải xác suất 100%?
3. Khi thêm nhiều điểm đến mới vào corpus, IDF và min-max score có thể thay đổi như thế nào?


In [ ]:
# TODO 3: In top 5 điểm đến cho mỗi feature
for j, feature in enumerate(features):
    # Gợi ý: dùng np.argsort(scores[:, j])[::-1][:5] để lấy index top 5
    top_idx = None  # ← thay None bằng code của bạn
    print(f"{feature}: " + ", ".join(places[i] for i in top_idx))
